# 241: Activity Brain Maps

### “what this script does” summary (for people running it)

- Loads ERSP samples and metadata from the 230 run, and uses the recon outputs (fsaverage-transformed coordinates) from the 240 run.

- Selects a time–frequency window (or loops over many windows) and optionally filters by condition.

- Aggregates ERSP activity per contact within that window (e.g., mean over the window, then mean/median across samples per contact).

- Projects contact-level activity onto the fsaverage cortical surface using spatial smoothing and a density/coverage weighting (to avoid over-trusting sparse regions).

- Saves view-based brain render images and a compact manifest/QC summary under a dedicated 241_atlas_activity_plots/ output tree so results are traceable to this script.

```
241_atlas_activity_plots/
  run230_<RUN_ID_230>__recon_<RUN_ID_RECON>/
    manifest.json
    frames/
      cond-picture/
        view-left/
          f000-002/
            t000-004.png
            t005-009.png
            ...
      cond-audio/
      cond-reading/
    videos/
      cond-picture_view-left_f000-002.mp4
      ...

```

##  Atlas input preparation (summary for readers)

### Purpose
This cell prepares all atlas-side inputs required for downstream visualization and reporting (240 reconstruction, 241 atlas activity plots). It converts sample-level clustering outputs into a consistent contact-level representation and maps contacts into fsaverage space.

### Inputs (what this cell reads and trusts)

- 230 clustering outputs

  - 230/.../<RUN_ID_230>/df_keep_with_clusters.parquet Sample-level table containing patient_id, electrode, condition, and cluster labels. This file defines which contacts exist and how samples map to contacts.

- Contact coordinate sources (patient space)

  - Paper1 contact tables / EL lookup spreadsheets / per-patient coordinate files. These provide one 3D coordinate per contact in native patient space.

- FreeSurfer assets

  - fsaverage surfaces and subject transforms used to map contacts into fsaverage tkrRAS space.

### What the cell does (core logic)

- Collapses sample-level rows into unique contact-level rows (patient_id × electrode).

- Resolves contact coordinates and transforms them into fsaverage tkrRAS space.

- Validates mapping consistency (counts, missing contacts, failed transforms).

- Writes a minimal, reusable atlas cache that downstream steps rely on.

### Outputs (what later notebooks consume)

- 240/.../<RUN_ID_RECON>/atlas_cache/df_meta_contact_level.tsv One row per contact with cluster membership and summary metadata.

- 240/.../<RUN_ID_RECON>/atlas_cache/electrode_coords_fsaverage_tkr.tsv Contact coordinates mapped into fsaverage space.

- 240/.../<RUN_ID_RECON>/atlas_cache/transform_qc_summary.tsv Per-patient QC table documenting transform success/failure and counts.

## - Key assumptions / constriants

  - Electrode naming must be consistent between df_keep_with_clusters.parquet and coordinate sources.

  - All downstream atlas visualizations assume *fsaverage tkrRAS* coordinates.

  - Contact-level outputs are reused across conditions; conditions differ only at the sample level.

In [6]:
import re
from pathlib import Path

import numpy as np
import functions.lf_blob_recon as R

# -------------------------
# CONFIG
# -------------------------
RUN_ID_230 = "20260111_191040"

CLUSTER_COL = "cluster_kmeans_101_k23_q0p9"  # used only to tag RUN_ID_RECON

FMAX_HZ = 500.0  # must match your ERSP frequency axis convention

# time window step (in time bins)
T_INC = 5

# optional: loop conditions; set to [None] for all samples pooled
CONDITIONS = ["picture", "audio", "reading"]  # e.g. [None, "picture", "audio", "reading"]

# frequency bands in Hz (low, high)
F_BANDS_HZ = [
    (0, 5),
    (5, 10),
    (10, 16),
    (16, 40),
    (40, 70),
    (70, 130),
    (130, 250),
    (250, 500),
]

# -------------------------
# Helpers
# -------------------------
def _safe_tag(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))
    return s.strip("_")

def hz_to_bin(hz: float, nF: int, fmax_hz: float) -> int:
    """Map Hz -> nearest frequency bin index in [0, nF-1], assuming linear 0..fmax_hz."""
    hz = float(hz)
    hz = max(0.0, min(hz, float(fmax_hz)))
    return int(round(hz / float(fmax_hz) * (nF - 1)))

def band_hz_to_bins(band_hz: tuple, nF: int, fmax_hz: float) -> tuple:
    """Convert (f_lo_hz, f_hi_hz) -> (f0_bin, f1_bin) inclusive, clipped."""
    f_lo, f_hi = float(band_hz[0]), float(band_hz[1])
    if f_lo > f_hi:
        f_lo, f_hi = f_hi, f_lo
    b0 = hz_to_bin(f_lo, nF, fmax_hz)
    b1 = hz_to_bin(f_hi, nF, fmax_hz)
    if b0 > b1:
        b0, b1 = b1, b0
    return (b0, b1)

# -------------------------
# RUN TAGGING
# -------------------------
CLUSTER_TAG = _safe_tag(CLUSTER_COL) if CLUSTER_COL else "auto_cluster"
RUN_ID_RECON = f"{RUN_ID_230}__{CLUSTER_TAG}"

# -------------------------
# Determine ERSP shape (nF, nT) from 230 run
# -------------------------
P = R.ensure_dirs(RUN_ID_230, RUN_ID_RECON)
run230_dir = Path(P["run230_dir"])

ersp_npy = run230_dir / "ersp_keep.npy"
ersp_npz = run230_dir / "ersp_keep.npz"

if ersp_npy.exists():
    ersp_shape = np.load(ersp_npy, mmap_mode="r").shape  # (n, nF, nT)
elif ersp_npz.exists():
    z = np.load(ersp_npz)
    # best effort: infer nF,nT from first key
    ersp_shape = z[z.files[0]].shape
    # if it is per-sample arrays, it might be (nF,nT); then n is len(files)
    if len(ersp_shape) == 2:
        ersp_shape = (len(z.files), ersp_shape[0], ersp_shape[1])
else:
    raise FileNotFoundError(f"Missing ERSP stack: {ersp_npy} or {ersp_npz}")

_, nF, nT = int(ersp_shape[0]), int(ersp_shape[1]), int(ersp_shape[2])
print(f"[QC] ERSP shape: nF={nF}, nT={nT}")

# Convert Hz bands -> bin bands (inclusive)
F_BANDS_BINS = [band_hz_to_bins(b, nF=nF, fmax_hz=FMAX_HZ) for b in F_BANDS_HZ]
print("[QC] F_BANDS_BINS:", F_BANDS_BINS)

# -------------------------
# IMPORTANT: run build_atlas_inputs ONCE if coords cache not present
# (render_surface_activity_from_ersp_window needs coords_out under 240 atlas_cache)
# -------------------------
coords_path = Path(P["coords_out"])
if not coords_path.exists():
    print("[INFO] coords_out missing; running build_atlas_inputs(...) once.")
    R.build_atlas_inputs(RUN_ID_230, RUN_ID_RECON, cluster_col=CLUSTER_COL)



[QC] ERSP shape: nF=129, nT=300
[QC] F_BANDS_BINS: [(0, 1), (1, 3), (3, 4), (4, 10), (10, 18), (18, 33), (33, 64), (64, 128)]


In [ ]:
# -------------------------
# Main loop: time windows × frequency bands × condition
# -------------------------
t_last = nT - 1

for cond in CONDITIONS:
    cond_tag = "all" if cond is None else str(cond)

    for t0 in range(0, nT, T_INC):
        t1 = min(t0 + T_INC - 1, t_last)

        for (b0, b1), (hz0, hz1) in zip(F_BANDS_BINS, F_BANDS_HZ):
            print(f"[RUN] cond={cond_tag}  t={t0:03d}-{t1:03d}  fbin={b0:03d}-{b1:03d}  fHz={hz0}-{hz1}")

            R.render_surface_activity_from_ersp_window(
                RUN_ID_230,                           # 230 run ID: where df_keep_with_clusters + ersp_keep are loaded from
                RUN_ID_RECON,                         # 240 recon run ID: where outputs/caches/PNGs are written

                f_bins=(b0, b1),                      # frequency window to aggregate (must match your function’s convention: bin indices or Hz)
                t_bins=(t0, t1),                      # time window to aggregate (usually bin indices in your ERSP grid)
                condition=cond,                       # restrict samples to this condition (None = use all)

                vmin=-6,                              # lower color limit for activity (more negative saturates to deepest blue)
                vmax=6,                               # upper color limit for activity (more positive saturates to deepest red)
                cmap_name="bwr",                      # colormap (bwr makes 0 map to white by design), "coolwarm" blue gray red

                k_nearest=4,                          # how many nearest contacts contribute to each surface vertex (larger = smoother/more mixing)
                sigma_mm=4.0,                         # spatial smoothing scale on the surface in mm (larger = blurrier, more cancellation toward 0)

                density_saturation=False,              # apply density-based modulation (low coverage darker/attenuated; high coverage less attenuated)
                density_gamma=0.6,                    # density contrast curve (<1 boosts low-density differences; >1 compresses them)
                s_min=0.5,                            # minimum density scaling factor (lower = sparse regions get darker/weaker)
                s_max=1.0,                            # maximum density scaling factor (higher = dense regions can appear stronger/brighter)
                exclude_contacts_dist_to_pial_mm_gt=12.0,  # drop contacts farther than this from pial (reduces deep contacts driving surface color)
                
                density_alpha = True,
                alpha_min = 0.10,
                alpha_max = 1.00,
                alpha_gamma  = 0.70,
                base_brain_gray_rgb = (0.5, 0.5, 0.5),

            )



[RUN] cond=picture  t=000-004  fbin=000-001  fHz=0-5
[RUN] cond=picture  t=000-004  fbin=001-003  fHz=5-10
[RUN] cond=picture  t=000-004  fbin=003-004  fHz=10-16
[RUN] cond=picture  t=000-004  fbin=004-010  fHz=16-40
[RUN] cond=picture  t=000-004  fbin=010-018  fHz=40-70
[RUN] cond=picture  t=000-004  fbin=018-033  fHz=70-130
[RUN] cond=picture  t=000-004  fbin=033-064  fHz=130-250
[RUN] cond=picture  t=000-004  fbin=064-128  fHz=250-500
[RUN] cond=picture  t=005-009  fbin=000-001  fHz=0-5
[RUN] cond=picture  t=005-009  fbin=001-003  fHz=5-10
[RUN] cond=picture  t=005-009  fbin=003-004  fHz=10-16
[RUN] cond=picture  t=005-009  fbin=004-010  fHz=16-40
[RUN] cond=picture  t=005-009  fbin=010-018  fHz=40-70
[RUN] cond=picture  t=005-009  fbin=018-033  fHz=70-130
[RUN] cond=picture  t=005-009  fbin=033-064  fHz=130-250
[RUN] cond=picture  t=005-009  fbin=064-128  fHz=250-500
[RUN] cond=picture  t=010-014  fbin=000-001  fHz=0-5
[RUN] cond=picture  t=010-014  fbin=001-003  fHz=5-10
[RUN] con

In [7]:
# Cell 2 — Build MP4 videos from the already-rendered 241 frames (robust writer)
from pathlib import Path
import re
import numpy as np
import imageio  # v2 API (get_writer)
import imageio.v3 as iio

import functions.lf_blob_recon as R


def _t0_from_png_name(p: Path) -> int:
    """
    Expected filename: t000-004.png  -> returns 0
    Fallback: returns a large number so bad names go last.
    """
    m = re.search(r"t(\d+)-(\d+)\.png$", p.name)
    return int(m.group(1)) if m else 10**9


def build_videos_for_run_241(
    run_id_230: str,
    run_id_recon: str = None,
    *,
    fps: int = 12,
    overwrite: bool = False,
):
    """
    Reads frames under:
      241/.../run230_<rid230>__recon_<ridrec>/frames/cond-*/view-*/f###-###/t###-###.png

    Writes MP4s to:
      241/.../run230_<rid230>__recon_<ridrec>/videos/
        cond-<cond>_view-<view>_f###-###.mp4
    """
    P241 = R.ensure_dirs_241(run_id_230, run_id_recon)
    frames_root = Path(P241["frames_root"])
    videos_root = Path(P241["videos_root"])
    videos_root.mkdir(parents=True, exist_ok=True)

    if not frames_root.exists():
        raise FileNotFoundError(f"Missing frames_root: {frames_root}")

    cond_dirs = sorted([p for p in frames_root.glob("cond-*") if p.is_dir()])
    if not cond_dirs:
        raise RuntimeError(f"No cond-* folders found under: {frames_root}")

    for cond_dir in cond_dirs:
        cond = cond_dir.name  # e.g. cond-picture
        view_dirs = sorted([p for p in cond_dir.glob("view-*") if p.is_dir()])
        for view_dir in view_dirs:
            view = view_dir.name  # e.g. view-left
            f_dirs = sorted([p for p in view_dir.glob("f*-*") if p.is_dir()])

            for f_dir in f_dirs:
                ftag = f_dir.name  # e.g. f000-002
                out_mp4 = videos_root / f"{cond}_{view}_{ftag}.mp4"

                if out_mp4.exists() and not overwrite:
                    print(f"[SKIP] {out_mp4}")
                    continue

                frames = sorted(
                    [p for p in f_dir.glob("*.png") if p.is_file()],
                    key=_t0_from_png_name
                )
                if not frames:
                    continue

                # First frame defines target size
                first = iio.imread(frames[0])
                if first.ndim == 2:
                    first = np.stack([first] * 3, axis=-1)
                if first.shape[-1] == 4:
                    first = first[:, :, :3]
                H, W = int(first.shape[0]), int(first.shape[1])

                def _pad_rgb(im):
                    if im.ndim == 2:
                        im = np.stack([im] * 3, axis=-1)
                    if im.shape[-1] == 4:
                        im = im[:, :, :3]
                    h, w = int(im.shape[0]), int(im.shape[1])
                    if (h, w) == (H, W):
                        return im
                    canvas = np.zeros((H, W, 3), dtype=im.dtype)
                    canvas[:min(H, h), :min(W, w), :] = im[:min(H, h), :min(W, w), :3]
                    return canvas

                try:
                    with imageio.get_writer(
                        str(out_mp4),
                        fps=int(fps),
                        codec="libx264",
                        quality=8,
                        macro_block_size=None,  # avoids resizing warnings for odd sizes
                    ) as w:
                        w.append_data(_pad_rgb(first))
                        for p in frames[1:]:
                            w.append_data(_pad_rgb(iio.imread(p)))

                    print(f"[WROTE] {out_mp4} (n={len(frames)})")

                except Exception as e:
                    raise RuntimeError(
                        f"Failed writing {out_mp4}.\n"
                        f"Likely causes:\n"
                        f"  - ffmpeg not available to imageio\n"
                        f"  - cannot write mp4 to the network path (permissions/locks)\n"
                        f"Original error: {e}"
                    )


# --- Run it (after your frame-generation loops finished) ---
build_videos_for_run_241(RUN_ID_230, RUN_ID_RECON, fps=10, overwrite=False)


[SKIP] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\241_atlas_activity_plots\run230_20260111_191040__recon_20260111_191040__cluster_kmeans_101_k23_q0p9\videos\cond-audio_view-dorsal_f000-001.mp4
[SKIP] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\241_atlas_activity_plots\run230_20260111_191040__recon_20260111_191040__cluster_kmeans_101_k23_q0p9\videos\cond-audio_view-dorsal_f001-003.mp4
[SKIP] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\241_atlas_activity_plots\run230_20260111_191040__recon_20260111_191040__cluster_kmeans_101_k23_q0p9\videos\cond-audio_view-dorsal_f003-004.mp4
[SKIP] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_Lora\02_FBM_Clustering\outputs\241_atlas_activity_plots\run230_20260111_191040__recon_20260111_191040__cluster_kmeans_101_k23_q0p9\videos\cond-audio_view-dorsal_f004-010.mp4
[SKIP] \\nasac-m2.unige.ch\m-HumanNeuron